# Baseline classifier

Both sets of Perch embeddings are now saved, so this notebook loads them and never re-embeds. The focal embeddings (clean single-species clips) are the training data. The soundscape embeddings (multi-species field segments) are the evaluation data.

A separate binary detector is trained per species, each answering "is this species present". This one-vs-rest design fits the mismatch between single-label training clips and multi-label field segments, and it is a standard, defensible baseline.

In [1]:
import numpy as np
import json

# focal = training (clean clips), soundscape = evaluation (field segments)
focal_emb = np.load('../data/focal_embeddings.npy')
with open('../data/focal_labels.json') as f:
    focal_labels = json.load(f)

ss_emb = np.load('../data/soundscape_embeddings.npy')
with open('../data/full_meta.json') as f:
    meta = json.load(f)
ss_labels = meta['labels']
ss_sites = meta['sites']

print("focal (train):", focal_emb.shape, "| species:", len(set(focal_labels)))
print("soundscape (eval):", ss_emb.shape, "| segments:", len(ss_labels))

focal (train): (1445, 1536) | species: 47
soundscape (eval): (1478, 1536) | segments: 1478


## Aligning the label space

Training covers 47 species, so the classifier can only predict those 47. The soundscape truth is built over the same 47 columns in a fixed order, so predictions and truth line up exactly. Species heard in the field but absent from training, including the 28 untrainable ones, cannot be predicted by this baseline and are excluded from its scoring, which is itself part of the finding.

In [2]:
from sklearn.preprocessing import MultiLabelBinarizer

species = sorted(set(focal_labels))            # the 47 trainable species
mlb = MultiLabelBinarizer(classes=species)

y_train = mlb.fit_transform([[l] for l in focal_labels])   # focal is single-label
y_eval = mlb.transform(ss_labels)                          # soundscape is multi-label

print("train matrix:", y_train.shape)
print("eval matrix:", y_eval.shape)
print("classes:", len(species))

train matrix: (1445, 47)
eval matrix: (1478, 47)
classes: 47


c:\dev\biodiversity\mapping-biodiversity-from-sound\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['1491113', '25073', '47158son01', '47158son02', '47158son03', '47158son04', '47158son05', '47158son06', '47158son07', '47158son08', '47158son09', '47158son10', '47158son11', '47158son12', '47158son13', '47158son14', '47158son15', '47158son16', '47158son17', '47158son18', '47158son19', '47158son20', '47158son21', '47158son22', '47158son23', '47158son24', '47158son25', '517063'] will be ignored
  warnings.warn(


## Training the one-vs-rest classifier

A logistic regression is trained per species on the focal embeddings, wrapped in a one-vs-rest scheme so each species gets its own binary detector. Logistic regression on frozen embeddings is a deliberately simple baseline, since the point is to measure what Perch's representation alone achieves before any heavier model is tried. The embeddings are standardised first, which logistic regression benefits from.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

scaler = StandardScaler()
X_train = scaler.fit_transform(focal_emb)
X_eval = scaler.transform(ss_emb)

clf = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, class_weight='balanced'),
    n_jobs=-1,
)
clf.fit(X_train, y_train)

# probability per species for each soundscape segment
y_prob = clf.predict_proba(X_eval)
print("predictions:", y_prob.shape)

predictions: (1478, 47)


## Scoring against field truth

The predictions are scored with the same metrics built in the evaluation harness: macro ROC-AUC across species, with average precision alongside. Only species with both positive and negative examples in the field data can be scored, so the number of species actually contributing to the average is reported next to it.

In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score

# species scoreable in the field: present in some segments but not all
pos = y_eval.sum(axis=0)
mask = (pos > 0) & (pos < y_eval.shape[0])

aucs, aps = [], []
for i in np.where(mask)[0]:
    aucs.append(roc_auc_score(y_eval[:, i], y_prob[:, i]))
    aps.append(average_precision_score(y_eval[:, i], y_prob[:, i]))

print("species scored:", int(mask.sum()))
print("macro ROC-AUC:", round(np.mean(aucs), 4))
print("macro avg precision:", round(np.mean(aps), 4))

species scored: 47
macro ROC-AUC: 0.8098
macro avg precision: 0.3793


## Baseline result

One-vs-rest logistic regression on standardised Perch embeddings, trained on clean focal clips for 47 species and evaluated on field soundscapes never seen in training.

- Macro ROC-AUC: 0.81 across 47 scored species
- Macro average precision: 0.38

The high ROC-AUC shows Perch embeddings carry strong species-relevant signal even under the focal-to-field domain shift. The much lower average precision reflects the severe class imbalance in the field data: most species are absent from most segments, so ranking presence above absence is achievable while making confident, precise detections is not. This gap motivates both a stronger model and careful per-species thresholding in later work. This baseline is the reference point all subsequent models are measured against.

## Where the baseline works and where it fails

The single macro score hides the thing this project is about. Broken down by taxonomic group and by site, it should show whether the baseline leans on the abundant bird data and struggles on the groups and places with less to go on. That unevenness, not the headline number, is the reliability story.

In [ ]:
import pandas as pd

tax = pd.read_csv('../data/taxonomy.csv')
group_map = dict(zip(tax['primary_label'].astype(str), tax['class_name']))

# per-species scores over the scoreable species
rows = []
for i in np.where(mask)[0]:
    rows.append({
        'species': species[i],
        'group': group_map.get(species[i], 'Unknown'),
        'roc_auc': roc_auc_score(y_eval[:, i], y_prob[:, i]),
        'avg_precision': average_precision_score(y_eval[:, i], y_prob[:, i]),
        'n_positive': int(y_eval[:, i].sum()),
    })
per_species = pd.DataFrame(rows)

# per-group scores over the scoreable species
by_group = (per_species.groupby('group')
            .agg(n_species=('species', 'count'),
                 mean_roc_auc=('roc_auc', 'mean'),
                 mean_avg_precision=('avg_precision', 'mean'))
            .round(3)
            .sort_values('mean_roc_auc', ascending=False))
print(by_group)

          n_species  mean_roc_auc  mean_avg_precision
group                                                
Aves             28         0.838               0.322
Amphibia         14         0.809               0.510
Mammalia          4         0.708               0.417
Reptilia          1         0.438               0.017


In [ ]:
# per-site: rebuild truth and predictions restricted to each site's segments
ss_sites = np.array(ss_sites)

site_rows = []

# per-site scores over the scoreable species
for site in sorted(set(ss_sites)):
    idx = np.where(ss_sites == site)[0]
    yt, yp = y_eval[idx], y_prob[idx]
    pos = yt.sum(axis=0)
    m = (pos > 0) & (pos < yt.shape[0])
    if m.sum() == 0:
        continue
    aucs = [roc_auc_score(yt[:, i], yp[:, i]) for i in np.where(m)[0]]
    site_rows.append({
        'site': site,
        'n_segments': len(idx),
        'n_species_scored': int(m.sum()),
        'macro_roc_auc': round(np.mean(aucs), 3),
    })

by_site = pd.DataFrame(site_rows)
print(by_site)

  site  n_segments  n_species_scored  macro_roc_auc
0  S03          48                 2          0.712
1  S08         120                 2          0.708
2  S09          38                 5          0.749
3  S13          48                 4          0.841
4  S15          96                11          0.841
5  S18          30                 3          0.639
6  S19          72                 6          0.718
7  S22         954                27          0.768
8  S23          72                 7          0.882


## Breakdown findings

Performance is uneven across both taxonomy and space, which is the core of the reliability picture.

By group, ROC-AUC falls with training abundance: birds 0.84, amphibians 0.81, mammals 0.71, the single reptile 0.44. Amphibians holding up despite minimal focal training reflects Perch's multi-taxa pretraining, and their average precision (0.51) exceeds birds' (0.32), so amphibian detections are more often correct even if slightly worse ranked. The reptile score rests on one species and is effectively chance.

By site, ROC-AUC ranges from 0.64 to 0.88. The best-sampled sites (S23, S15)